Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\Desktop\\pruebas_collab\\datosNarmax\\1pasos_mlp_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [3]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 1
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52404, 12, 6)
Dimensiones de Y: (52404, 1)


In [9]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [10]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (52404, 72)


Se dividen nuevamente los conjuntos de datos

In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36682, 72)
Las dimensiones de testX son:  (10533, 72)
Las dimensiones de valX son:  (5189, 72)


In [12]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36682, 1)
Las dimensiones de testY son:  (10533, 1)
Las dimensiones de valY son:  (5189, 1)


Se crean métricas para medir desempeño

In [13]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=params['epochs'],
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])

    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

231/231 - 4s - 16ms/step - ia: 0.3727 - loss: 1.5998 - mae: 0.9174 - rmse: 1.2188 - smape: 1.3176 - val_ia: 0.2918 - val_loss: 0.5030 - val_mae: 0.6025 - val_rmse: 0.6558 - val_smape: 1.1774

Epoch 2/128                                           

231/231 - 1s - 2ms/step - ia: 0.4732 - loss: 0.7667 - mae: 0.6591 - rmse: 0.8643 - smape: 1.1784 - val_ia: 0.3734 - val_loss: 0.3105 - val_mae: 0.4665 - val_rmse: 0.5154 - val_smape: 0.9256

Epoch 3/128                                           

231/231 - 1s - 3ms/step - ia: 0.5286 - loss: 0.5931 - mae: 0.5803 - rmse: 0.7600 - smape: 1.1005 - val_ia: 0.4153 - val_loss: 0.2505 - val_mae: 0.4196 - val_rmse: 0.4631 - val_smape: 0.8263

Epoch 4/128                                           

231/231 - 1s - 3ms/step - ia: 0.5743 - loss: 0.4860 - mae: 0.5228 - rmse: 0.6874 - smape: 1.0238 - val_ia: 0.4337 - val_loss: 0.2259 - val_mae: 0.4018 - val_rmse: 0.4407 - val_smape: 0.7811

Epoch 5/128

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



29/29 - 2s - 78ms/step - ia: 0.7220 - loss: 0.2159 - mae: 0.3334 - rmse: 0.4194 - smape: 0.7005 - val_ia: 0.7607 - val_loss: 0.1213 - val_mae: 0.2905 - val_rmse: 0.3388 - val_smape: 0.6403

Epoch 2/16                                                                        

29/29 - 0s - 5ms/step - ia: 0.9119 - loss: 0.0305 - mae: 0.1295 - rmse: 0.1717 - smape: 0.3378 - val_ia: 0.8801 - val_loss: 0.0400 - val_mae: 0.1568 - val_rmse: 0.1947 - val_smape: 0.3782

Epoch 3/16                                                                        

29/29 - 0s - 5ms/step - ia: 0.9434 - loss: 0.0135 - mae: 0.0837 - rmse: 0.1153 - smape: 0.2378 - val_ia: 0.9259 - val_loss: 0.0182 - val_mae: 0.0997 - val_rmse: 0.1321 - val_smape: 0.2883

Epoch 4/16                                                                        

29/29 - 0s - 6ms/step - ia: 0.9557 - loss: 0.0086 - mae: 0.0656 - rmse: 0.0926 - smape: 0.1953 - val_ia: 0.9350 - val_loss: 0.0143 - val_mae: 0.0891 - val_rmse: 0.1169 - val_smape:

...

231/231 - 3s - 13ms/step - ia: 0.3271 - loss: 2.7437 - mae: 1.2468 - rmse: 1.6364 - smape: 1.3796 - val_ia: 0.2693 - val_loss: 0.6453 - val_mae: 0.6582 - val_rmse: 0.7418 - val_smape: 1.0979

Epoch 2/64                                                                         

231/231 - 1s - 3ms/step - ia: 0.3269 - loss: 2.6817 - mae: 1.2326 - rmse: 1.6175 - smape: 1.3785 - val_ia: 0.2689 - val_loss: 0.6456 - val_mae: 0.6590 - val_rmse: 0.7423 - val_smape: 1.1016

Epoch 3/64                                                                         

231/231 - 1s - 2ms/step - ia: 0.3378 - loss: 2.5779 - mae: 1.1995 - rmse: 1.5821 - smape: 1.3634 - val_ia: 0.2684 - val_loss: 0.6461 - val_mae: 0.6599 - val_rmse: 0.7428 - val_smape: 1.1053

Epoch 4/64                                                                         

231/231 - 0s - 2ms/step - ia: 0.3338 - loss: 2.5434 - mae: 1.1997 - rmse: 1.5744 - smape: 1.3762 - val_ia: 0.2679 - val_loss: 0.6468 - val_mae: 0.6609 - val_rmse: 0.7434 -

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



461/461 - 3s - 7ms/step - ia: 0.7151 - loss: 0.3282 - mae: 0.4288 - rmse: 0.5579 - smape: 0.7283 - val_ia: 0.4318 - val_loss: 0.1238 - val_mae: 0.2386 - val_rmse: 0.2998 - val_smape: 0.4689

Epoch 2/32                                                                         

461/461 - 1s - 2ms/step - ia: 0.7779 - loss: 0.1958 - mae: 0.3166 - rmse: 0.4296 - smape: 0.5986 - val_ia: 0.5231 - val_loss: 0.0836 - val_mae: 0.1809 - val_rmse: 0.2431 - val_smape: 0.3735

Epoch 3/32                                                                         

461/461 - 1s - 2ms/step - ia: 0.8122 - loss: 0.1434 - mae: 0.2656 - rmse: 0.3658 - smape: 0.5403 - val_ia: 0.5321 - val_loss: 0.0701 - val_mae: 0.1671 - val_rmse: 0.2278 - val_smape: 0.3483

Epoch 4/32                                                                         

461/461 - 1s - 2ms/step - ia: 0.8321 - loss: 0.1131 - mae: 0.2332 - rmse: 0.3241 - smape: 0.4952 - val_ia: 0.5398 - val_loss: 0.0607 - val_mae: 0.1586 - val_rmse: 0.2156 - 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



922/922 - 5s - 5ms/step - ia: 0.6117 - loss: 0.3790 - mae: 0.4675 - rmse: 0.5672 - smape: 0.8693 - val_ia: 0.2972 - val_loss: 0.1404 - val_mae: 0.2970 - val_rmse: 0.3094 - val_smape: 0.4273

Epoch 2/16                                                                         

922/922 - 2s - 2ms/step - ia: 0.7724 - loss: 0.1369 - mae: 0.2890 - rmse: 0.3555 - smape: 0.5926 - val_ia: 0.3122 - val_loss: 0.1422 - val_mae: 0.2947 - val_rmse: 0.3068 - val_smape: 0.4186

Epoch 3/16                                                                         

922/922 - 2s - 2ms/step - ia: 0.7998 - loss: 0.1088 - mae: 0.2553 - rmse: 0.3168 - smape: 0.5390 - val_ia: 0.3196 - val_loss: 0.1220 - val_mae: 0.2774 - val_rmse: 0.2908 - val_smape: 0.4194

Epoch 4/16                                                                         

922/922 - 2s - 2ms/step - ia: 0.8144 - loss: 0.0937 - mae: 0.2372 - rmse: 0.2941 - smape: 0.5135 - val_ia: 0.3004 - val_loss: 0.1451 - val_mae: 0.3042 - val_rmse: 0.3178 - 

In [16]:
print(best)

{'activation': 2, 'batch': 0, 'dropout': 0.30000000000000004, 'epochs': 2, 'layers': 1.0, 'learning_rate': 0.00027534929346813096, 'units': 4}


In [17]:
print(best)

{'activation': 2, 'batch': 0, 'dropout': 0.30000000000000004, 'epochs': 2, 'layers': 1.0, 'learning_rate': 0.00027534929346813096, 'units': 4}
